# Anime Dubber v3 — Kaggle Notebook

Полный пайплайн перевода аниме:
1. Извлечение аудио
2. ASR (Whisper)
3. Перевод (Groq/OpenRouter с ключами)
4. Сепарация вокала от фона (Demucs)
5. TTS с разными голосами для персонажей
6. Правильный ducking
7. Сборка видео

## Настройки
Измени `INPUT_VIDEO` и `TARGET_LANG` в ячейке 2.

### API-ключи
Добавь ключи в Kaggle Secrets (Add-ons → Secrets):
- `GROQ_API_KEY`
- `OPENROUTER_API_KEY`

Или получи бесплатно:
- Groq: console.groq.com/keys
- OpenRouter: openrouter.ai/keys

In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"  # ru, en, ja, ko
SOURCE_LANG = "ja"  # ja, en

# === GET API KEYS FROM KAGGLE SECRETS ===
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")

# === INSTALL DEPS ===
!pip install -q faster-whisper edge-tts httpx demucs
!pip install -q pydub librosa soundfile numpy scipy

import os, json, asyncio, subprocess, shutil, sys
from pathlib import Path

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

WORK = Path("/kaggle/working")
JOB = WORK / "dub_v3"
JOB.mkdir(exist_ok=True)

print(f"Input: {INPUT_VIDEO}")
print(f"Exists: {Path(INPUT_VIDEO).exists()}")

In [ ]:
# === STAGE 1: Extract Audio ===
audio_path = JOB / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", INPUT_VIDEO,
    "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2",
    str(audio_path)
], check=True, capture_output=True)
print(f"Audio: {audio_path.stat().st_size / 1024:.0f} KB")

In [ ]:
# === STAGE 2: ASR ===
from faster_whisper import WhisperModel
model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
segments, info = model.transcribe(str(audio_path), language=SOURCE_LANG, beam_size=5, word_timestamps=True)
seg_list = [{"id": f"seg_{i:03d}", "start": s.start, "end": s.end, "text": s.text.strip()}
            for i, s in enumerate(segments)]
(JOB / "asr.json").write_text(json.dumps(seg_list, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"ASR: {len(seg_list)} segments")

In [ ]:
# === STAGE 3: Translate (LLM with fallback) ===
import httpx

def translate_texts(texts, src_lang, tgt_lang):
    """Translate using Groq with OpenRouter fallback."""
    if not texts:
        return texts
    
    lang_names = {"ja": "Japanese", "en": "English", "ko": "Korean", "zh": "Chinese", "ru": "Russian"}
    numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(texts))
    prompt = f"""Translate the following {lang_names.get(src_lang, src_lang)} manga dialogue lines to natural {lang_names.get(tgt_lang, tgt_lang)}.
Keep the tone and style. Return ONLY a JSON array of strings, same order, no explanation.

{numbered}"""
    
    # Try Groq first
    try:
        with httpx.Client(timeout=30) as c:
            r = c.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
                json={"model": "llama-3.3-70b-versatile", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}
            )
            r.raise_for_status()
            result = r.json()["choices"][0]["message"]["content"]
            return _parse_json_array(result, len(texts))
    except Exception as e:
        print(f"Groq failed: {e}, trying OpenRouter...")
    
    # Fallback to OpenRouter
    try:
        with httpx.Client(timeout=30) as c:
            r = c.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
                json={"model": "google/gemini-2.0-flash-001", "messages": [{"role": "user", "content": prompt}]}
            )
            r.raise_for_status()
            result = r.json()["choices"][0]["message"]["content"]
            return _parse_json_array(result, len(texts))
    except Exception as e:
        print(f"OpenRouter failed: {e}")
        return texts

def _parse_json_array(text, expected_len):
    try:
        arr = json.loads(text)
        if len(arr) == expected_len:
            return arr
    except:
        pass
    import re
    match = re.search(r'\[[\s\S]*?\]', text)
    if match:
        try:
            arr = json.loads(match.group())n            if len(arr) == expected_len:
                return arr
        except:
            pass
    return []

texts = [s["text"] for s in seg_list]
translations = translate_texts(texts, SOURCE_LANG, TARGET_LANG)
for seg, tr in zip(seg_list, translations):
    seg["translation"] = tr
print(f"Translated: {len(translations)} lines")

In [ ]:
# === STAGE 4: Separate vocals from background ===
print("Separating vocals from background...")
vocals_path = JOB / "vocals.wav"
background_path = JOB / "background.wav"
if not vocals_path.exists() or not background_path.exists():
    subprocess.run(["python", "-m", "demucs", "--two-stems", "vocals", "-o", str(JOB), str(audio_path)], check=True, capture_output=True, timeout=300)
    demucs_output = JOB / "htdemucs" / audio_path.stem
    if demucs_output.exists():
        shutil.move(str(demucs_output / "vocals.wav"), str(vocals_path))
        shutil.move(str(demucs_output / "no_vocals.wav"), str(background_path))
        shutil.rmtree(str(demucs_output))
print(f"Vocals: {vocals_path.stat().st_size / 1024:.0f} KB")
print(f"Background: {background_path.stat().st_size / 1024:.0f} KB")

In [ ]:
# === STAGE 5: TTS with auto voice per segment ===
import edge_tts
VOICE_POOLS = {
    "ru": {"male": ["ru-RU-DmitryNeural", "ru-RU-YuriyNeural"], "female": ["ru-RU-SvetlanaNeural", "ru-RU-DariyaNeural"]},
    "en": {"male": ["en-US-GuyNeural", "en-US-DavisNeural"], "female": ["en-US-AriaNeural", "en-US-JennyNeural"]},
    "ja": {"male": ["ja-JP-KeitaNeural"], "female": ["ja-JP-NanamiNeural"]},
    "ko": {"male": ["ko-KR-InJoonNeural"], "female": ["ko-KR-SunHiNeural"]},
}
def get_voice(idx, lang):
    pool = VOICE_POOLS.get(lang, VOICE_POOLS["en"])
    voices = pool["male"] if idx % 2 == 0 else pool["female"]
    return voices[(idx // 2) % len(voices)]
tts_dir = JOB / "tts"
tts_dir.mkdir(exist_ok=True)
async def do_tts(text, voice, out):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out))
for i, seg in enumerate(seg_list):
    out = tts_dir / f"{seg['id']}.wav"
    if not out.exists():
        voice = get_voice(i, TARGET_LANG)
        await asyncio.to_thread(lambda: asyncio.run(do_tts(seg["translation"], voice, out)))
    seg["tts_path"] = str(out)
print(f"TTS: {len(seg_list)} files")

In [ ]:
# === STAGE 6: Mix with ducking ===
import numpy as np, soundfile as sf
from scipy.signal import resample
background, sr = sf.read(str(background_path))
if background.ndim > 1: background = background.mean(axis=1)
output = background.copy().astype(np.float64)
duck_factor = 10 ** (-15/20)
attack = int(0.05 * sr)
release = int(0.3 * sr)
for seg in seg_list:
    start, end = int(seg["start"]*sr), int(seg["end"]*sr)
    env = np.ones(end - start)
    if len(env) > attack: env[:attack] = np.linspace(1.0, duck_factor, attack)
    if len(env) > release: env[-release:] = np.linspace(duck_factor, 1.0, release)
    env[attack:-release] = duck_factor
    output[start:end] *= env
    tts, tts_sr = sf.read(seg["tts_path"])
    if tts.ndim > 1: tts = tts.mean(axis=1)
    if tts_sr != sr: tts = resample(tts, int(len(tts) * sr / tts_sr))
    mix_len = min(end - start, len(tts))
    output[start:start+mix_len] += tts[:mix_len]
output = output / np.max(np.abs(output)) * 0.95
out_path = JOB / "output.wav"
sf.write(str(out_path), output.astype(np.float32), sr)
print(f"Done: {out_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# === STAGE 7: Combine video + audio ===
final_video = JOB / "output.mp4"
subprocess.run(["ffmpeg", "-y", "-i", INPUT_VIDEO, "-i", str(out_path), "-c:v", "copy", "-map", "0:v:0", "-map", "1:a:0", "-shortest", str(final_video)], check=True, capture_output=True)
print(f"Final: {final_video.stat().st_size / 1024 / 1024:.2f} MB")
from IPython.display import FileLink
FileLink(str(final_video))